In [ ]:
# Import basic libraries
import matplotlib.pyplot as plt
import random
import numpy as np
import torch

# set common seed for all libraries (Same seed as Assignment 3)
random.seed(24)
np.random.seed(24)
torch.manual_seed(24)
torch.cuda.manual_seed_all(24)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Task 1: MNIST Setup:

## Import data

In [ ]:
from torchvision import datasets, transforms

In [ ]:
# import MNIST test dataset
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)


## Preprocess and Split Data

In [ ]:
# seperate images and labels from dataset and preprocess images
images = []
labels = []

for i in range(len(dataset)):
    image, label = dataset[i]
    # round images to map them into zero and one
    images.append(np.round(image))
    labels.append(label)

images = torch.stack(images).to(torch.float32)
labels = torch.tensor(labels)

In [ ]:
# Collect train and test dataset
X_train = images[:1000]
y_train = labels[:1000]

X_test = images[1000:2000]
y_test = labels[1000:2000]


In [ ]:
# check data type
X_train.dtype, X_test.dtype

In [ ]:
# show train images
fig, axes = plt.subplots(4,4, figsize=(4,4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i].reshape(28,28), cmap='gray')
    ax.axis('off')

plt.suptitle('Training Images')
plt.tight_layout()
plt.show()


## Data Loader


In [ ]:
from torch.utils.data import DataLoader


In [ ]:
# create a DataLoader for the train set that loads X_train
train_loader = DataLoader(X_train, batch_size=64, shuffle=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# Task 2: Simple Diffusion Forward Process:
- Implement the forward (noise-adding) process for T=1000 7mesteps.
- Visualize the progressive noising of a sample digit at different 7mesteps (t=0, 250, 500,
750, 1000).
- Why do we add Gaussian noise gradually instead of all at once?

In [ ]:
T = 1000
# linear beta schedule (simple + fine for MNIST)
betas = torch.linspace(1e-4, 2e-2, T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise=None):
    """
    x0: [B,1,28,28] clean images
    t : [B] timesteps in 1..T
    noise: optional, else sampled ~ N(0,1)
    returns x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*eps
    """
    if noise is None:
        noise = torch.randn_like(x0)
    a_bar_t = alpha_bar[t-1].view(-1,1,1,1)     # gather per sample
    return a_bar_t.sqrt()*x0 + (1.0 - a_bar_t).sqrt()*noise


In [ ]:
import matplotlib.pyplot as plt

# get one clean image
x0 = X_train[0:1].to(device)                    # [1,1,28,28]

ts = torch.tensor([1,50, 250, 500, 750, 1000], device=device)
x_t = q_sample(x0.repeat(ts.numel(),1,1,1), ts) # make a copy of data and add each noise step to a copy

imgs = [x0.cpu()] + [x_t[i:i+1].cpu() for i in range(len(ts))]
titles = ["t=0","t=1","t=50","t=250","t=500","t=750","t=1000"]

plt.figure(figsize=(9,2))
for i,(im,tt) in enumerate(zip(imgs, titles), 1):
    plt.subplot(1,len(imgs),i)
    plt.imshow(im[0,0], cmap='gray'); plt.axis("off"); plt.title(tt)
plt.tight_layout(); plt.show()
